# Notebook 05 — Portfolio Backtest: Long/Short Dollar-Neutral Strategy

**Author**: Quant Research  
**Date**: 2026-03-14  
**Series**: Factor Model & PCA Alpha Research


## Executive Summary

This notebook runs the full quantitative backtest of the composite alpha signal
developed in Notebook 04. The strategy is a **dollar-neutral long/short equity**
portfolio with:

- **Universe**: ~80–100 large-cap US equities (survivorship bias disclaimer below)
- **Signal**: IC-weighted composite of 12-1 momentum and 1-month reversal (Notebook 04)
- **Construction**: Quintile sort → long Q5, short Q1; equal-weighted within each leg
- **Rebalancing**: Monthly (end-of-month)
- **Transaction costs**: 5 bps one-way (conservative for liquid large-caps)
- **Risk**: No leverage constraint; dollar-neutral maintained each period

**Reported metrics**: Gross Sharpe, Net Sharpe (after costs), Max Drawdown, Calmar,
IC, Turnover. Gross and net Sharpe are reported separately — the spread reveals
the cost of turnover.

**Survivorship bias warning**: The universe used here is a proxy for large-cap US
equities using current constituents or a simulated panel. This introduces survivorship
bias that inflates backtest results. Results should be discounted accordingly.
A proper implementation would use CRSP/Compustat point-in-time data (available via WRDS).

**Parameter discipline**: No parameters were optimized post-hoc to maximize Sharpe.
The 5 bps cost, monthly rebalance, and quintile sort are standard choices from the
literature. Report what you get honestly.

**References**:  
Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*, 2nd ed. McGraw-Hill.  
Jegadeesh, N. & Titman, S. (1993). *Journal of Finance*, 48(1), 65–91.


## 1. Imports and Configuration


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

SRC_AVAILABLE = False
try:
    from src.data.loader import fetch_returns
    from src.data.universe import get_universe
    from src.factors.technical import momentum_12_1, short_term_reversal
    from src.backtest.engine import run_backtest, quintile_backtest
    from src.backtest.portfolio import build_portfolio_weights
    from src.backtest.metrics import performance_metrics, format_metrics_table
    from src.analysis.tearsheet import generate_tearsheet
    SRC_AVAILABLE = True
    print("[OK] src modules loaded")
except ImportError as e:
    print(f"[WARN] src import failed: {e} — using inline implementations")

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

START = "2010-01-01"
END   = "2023-12-31"
ONE_WAY_COST_BPS = 5.0   # conservative for liquid large-caps
QUINTILE_LONG    = 5      # top quintile
QUINTILE_SHORT   = 1      # bottom quintile
REBAL_FREQ_DAYS  = 21     # monthly rebalancing

print(f"Config: cost={ONE_WAY_COST_BPS}bps, rebalance=monthly, universe=large-cap proxy")


## 2. Data Load and Signal Construction


In [ ]:
def winsorize_zscore(series, n_std=3.0):
    mu, sigma = series.mean(), series.std()
    clipped = series.clip(lower=mu - n_std*sigma, upper=mu + n_std*sigma)
    s = clipped.std()
    return (clipped - clipped.mean()) / s if s > 0 else clipped * 0


SYNTHETIC = False
if SRC_AVAILABLE:
    try:
        universe = get_universe()
        returns = fetch_returns(universe, start=START, end=END)
        print(f"[OK] Live returns: {returns.shape}")
    except Exception as e:
        print(f"[WARN] {e} — synthetic")
        SYNTHETIC = True
else:
    SYNTHETIC = True

if SYNTHETIC:
    rng = np.random.default_rng(42)
    idx = pd.bdate_range(START, END)
    N = 80
    base = rng.normal(3e-4, 0.012, (len(idx), N))
    for t in range(126, len(idx)):
        past = base[t-126:t].sum(axis=0)
        base[t] += 0.003 * (past - past.mean()) / (past.std() + 1e-8)
    returns = pd.DataFrame(base, index=idx, columns=[f'S{i:03d}' for i in range(N)])
    print("[SYNTHETIC DATA]")

returns = returns.dropna(axis=1, thresh=int(0.9*len(returns)))

# Build composite signal
if SRC_AVAILABLE:
    try:
        mom_sig = momentum_12_1(returns)
        rev_sig = short_term_reversal(returns)
    except Exception:
        log_r = np.log(1 + returns)
        mom_sig = log_r.rolling(231).sum().shift(21).apply(winsorize_zscore, axis=1)
        rev_sig = (-log_r.rolling(21).sum()).apply(winsorize_zscore, axis=1)
else:
    log_r = np.log(1 + returns)
    mom_sig = log_r.rolling(231).sum().shift(21).apply(winsorize_zscore, axis=1)
    rev_sig = (-log_r.rolling(21).sum()).apply(winsorize_zscore, axis=1)

# Compute IC for weighting
fwd_21 = returns.shift(-21).rolling(21).sum()
monthly_idx = pd.bdate_range(START, END, freq='BM')
monthly_idx = monthly_idx[monthly_idx.isin(mom_sig.index)]

ic_ests = {}
for sig_name, sig_df in [('mom', mom_sig), ('rev', rev_sig)]:
    ics = []
    for d in monthly_idx[:-1]:
        if d not in fwd_21.index:
            continue
        sr = sig_df.loc[d].dropna()
        fr = fwd_21.loc[d].dropna()
        c = sr.index.intersection(fr.index)
        if len(c) >= 20:
            rho, _ = spearmanr(sr[c], fr[c])
            ics.append(rho)
    ic_ests[sig_name] = np.nanmean(ics) if ics else 0.0

total_ic = ic_ests['mom'] + ic_ests['rev']
w_mom = ic_ests['mom'] / total_ic if total_ic != 0 else 0.5
w_rev = 1 - w_mom

print(f"IC weights: momentum={w_mom:.3f}  reversal={w_rev:.3f}")

composite = (w_mom * mom_sig + w_rev * rev_sig).apply(winsorize_zscore, axis=1)
print(f"Composite signal shape: {composite.shape}")


## 3. Backtest Engine — Quintile Sort, Long/Short Portfolio

At each monthly rebalance date:
1. Rank stocks by composite alpha signal
2. Long top quintile (Q5), short bottom quintile (Q1)
3. Equal-weight within each leg; normalize to dollar-neutral (long $ = short $)
4. Hold until next rebalance date
5. Compute realized return as long_leg - short_leg

**Transaction costs**: 5 bps one-way, applied at each rebalance on the turnover fraction.
Net return = gross return - 2 × 5bps × turnover_fraction.


In [ ]:
if SRC_AVAILABLE:
    try:
        results = quintile_backtest(
            composite, returns,
            n_quintiles=5,
            long_q=QUINTILE_LONG,
            short_q=QUINTILE_SHORT,
            one_way_cost_bps=ONE_WAY_COST_BPS
        )
        gross_returns = results['gross_returns']
        net_returns   = results['net_returns']
        turnover      = results['turnover']
        weights_hist  = results.get('weights', None)
        print("[OK] Backtest from src/")
    except Exception as e:
        print(f"[WARN] {e} — running inline backtest")
        SRC_AVAILABLE = False

if not SRC_AVAILABLE:
    # ── Inline backtest loop ─────────────────────────────────────────────────
    ONE_WAY_COST = ONE_WAY_COST_BPS / 10000

    gross_list, net_list, turnover_list, date_list = [], [], [], []
    prev_weights = pd.Series(dtype=float)

    for i, rebal_date in enumerate(monthly_idx[:-1]):
        next_date = monthly_idx[i + 1]
        if rebal_date not in composite.index:
            continue

        # Select signal row at rebalance date
        sig_row = composite.loc[rebal_date].dropna()
        if len(sig_row) < 20:
            continue

        # Quintile sort
        quintiles = pd.qcut(sig_row, 5, labels=[1,2,3,4,5])
        long_stocks  = quintiles[quintiles == 5].index
        short_stocks = quintiles[quintiles == 1].index

        n_long  = len(long_stocks)
        n_short = len(short_stocks)
        if n_long == 0 or n_short == 0:
            continue

        # Equal-weighted, dollar-neutral
        weights = pd.Series(0.0, index=sig_row.index)
        weights[long_stocks]  =  1.0 / n_long
        weights[short_stocks] = -1.0 / n_short

        # Compute turnover vs previous weights
        if len(prev_weights) > 0:
            all_stocks = weights.index.union(prev_weights.index)
            w_cur  = weights.reindex(all_stocks, fill_value=0)
            w_prev = prev_weights.reindex(all_stocks, fill_value=0)
            tov = (w_cur - w_prev).abs().sum() / 2
        else:
            tov = 1.0

        # Realized returns over holding period
        if next_date not in returns.index:
            continue
        hold_slice = returns.loc[rebal_date:next_date]
        if len(hold_slice) < 2:
            continue

        common = weights.index.intersection(hold_slice.columns)
        period_returns = (1 + hold_slice[common]).prod() - 1  # cumulative return over period
        gross_r = (weights[common] * period_returns).sum()
        cost    = 2 * ONE_WAY_COST * tov
        net_r   = gross_r - cost

        gross_list.append(gross_r)
        net_list.append(net_r)
        turnover_list.append(tov)
        date_list.append(rebal_date)
        prev_weights = weights

    gross_returns = pd.Series(gross_list, index=date_list, name='gross')
    net_returns   = pd.Series(net_list,   index=date_list, name='net')
    turnover      = pd.Series(turnover_list, index=date_list, name='turnover')

print(f"\nBacktest complete.")
print(f"Periods: {len(gross_returns)}")
print(f"Date range: {gross_returns.index[0].date()} → {gross_returns.index[-1].date()}")
print(f"Average monthly turnover: {turnover.mean():.2%}")


## 4. Performance Metrics — Gross vs. Net


In [ ]:
def calc_metrics(returns_series: pd.Series, freq: int = 12) -> dict:
    """
    Compute performance metrics assuming monthly return series.
    freq=12 for monthly → annual scaling.
    """
    r = returns_series.dropna()
    ann_ret = r.mean() * freq
    ann_vol = r.std() * np.sqrt(freq)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan

    # Drawdown
    cum = (1 + r).cumprod()
    rolling_max = cum.cummax()
    drawdown = (cum - rolling_max) / rolling_max
    max_dd = drawdown.min()

    calmar = ann_ret / abs(max_dd) if max_dd != 0 else np.nan

    # Sortino
    downside_ret = r[r < 0]
    downside_vol = downside_ret.std() * np.sqrt(freq) if len(downside_ret) > 1 else np.nan
    sortino = ann_ret / downside_vol if downside_vol and downside_vol > 0 else np.nan

    win_rate = (r > 0).mean()

    return {
        'Ann. Return (%)':   ann_ret  * 100,
        'Ann. Vol (%)':      ann_vol  * 100,
        'Sharpe Ratio':      sharpe,
        'Sortino Ratio':     sortino,
        'Max Drawdown (%)':  max_dd   * 100,
        'Calmar Ratio':      calmar,
        'Win Rate (%)':      win_rate * 100,
        'Avg Turnover (%)':  turnover.mean() * 100,
    }


if SRC_AVAILABLE:
    try:
        metrics_gross = performance_metrics(gross_returns)
        metrics_net   = performance_metrics(net_returns)
    except Exception:
        metrics_gross = calc_metrics(gross_returns)
        metrics_net   = calc_metrics(net_returns)
else:
    metrics_gross = calc_metrics(gross_returns)
    metrics_net   = calc_metrics(net_returns)

metrics_df = pd.DataFrame({'Gross': metrics_gross, 'Net (5bps)': metrics_net})
print("\n=== Performance Metrics — Gross vs. Net of Transaction Costs ===")
print(metrics_df.round(3).to_string())

sharpe_drag = metrics_gross.get('Sharpe Ratio', np.nan) - metrics_net.get('Sharpe Ratio', np.nan)
print(f"\nSharpe drag from transaction costs: {sharpe_drag:.3f}")
if sharpe_drag > 0.3:
    print("→ High cost drag — consider lower-turnover signal or higher cost assumptions.")
else:
    print("→ Modest cost drag — strategy relatively robust to 5 bps costs.")


## 5. Cumulative Returns — Gross vs. Net


In [ ]:
cum_gross = (1 + gross_returns).cumprod()
cum_net   = (1 + net_returns).cumprod()

fig, axes = plt.subplots(3, 1, figsize=(13, 12),
                          gridspec_kw={'height_ratios': [3, 1.5, 1.5]})

# Panel 1: Cumulative returns
ax = axes[0]
ax.plot(cum_gross.index, cum_gross.values, label='Gross', color='#1f77b4', linewidth=2.0)
ax.plot(cum_net.index,   cum_net.values,   label='Net (5 bps)', color='#2ca02c', linewidth=2.0)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
ax.set_ylabel('Growth of $1 (monthly rebalance)')
ax.set_title('Long/Short Dollar-Neutral Strategy — Cumulative Returns', fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Panel 2: Monthly returns
ax = axes[1]
colors_bar = ['#2ca02c' if r >= 0 else '#d62728' for r in net_returns.values]
ax.bar(net_returns.index, net_returns.values * 100, color=colors_bar,
       width=20, label='Monthly net return')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Monthly Return (%)')
ax.set_title('Monthly Net Returns', fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Panel 3: Drawdown
cum_net_full = (1 + net_returns).cumprod()
rolling_max_net = cum_net_full.cummax()
drawdown_net = (cum_net_full - rolling_max_net) / rolling_max_net * 100

ax = axes[2]
ax.fill_between(drawdown_net.index, drawdown_net.values, 0,
                color='#d62728', alpha=0.6, label='Drawdown')
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown — Net Strategy', fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../results/backtest_tearsheet.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Quintile Return Spread Analysis

We backtest all 5 quintile portfolios (long-only, zero cost) to verify monotonicity.
If Q5 > Q4 > Q3 > Q2 > Q1, the signal has consistent cross-sectional predictive power.


In [ ]:
quintile_returns = {q: [] for q in range(1, 6)}
q_dates = []

for i, rebal_date in enumerate(monthly_idx[:-1]):
    next_date = monthly_idx[i + 1]
    if rebal_date not in composite.index:
        continue
    sig_row = composite.loc[rebal_date].dropna()
    if len(sig_row) < 20:
        continue

    try:
        quintiles_labels = pd.qcut(sig_row, 5, labels=[1,2,3,4,5])
    except ValueError:
        continue

    if next_date not in returns.index:
        continue
    hold = returns.loc[rebal_date:next_date]
    if len(hold) < 2:
        continue
    period_ret = (1 + hold).prod() - 1

    for q in range(1, 6):
        q_stocks = quintiles_labels[quintiles_labels == q].index
        q_common = q_stocks.intersection(period_ret.index)
        if len(q_common) > 0:
            quintile_returns[q].append(period_ret[q_common].mean())
        else:
            quintile_returns[q].append(np.nan)
    q_dates.append(rebal_date)

q_df = pd.DataFrame(quintile_returns, index=q_dates)
q_cum = (1 + q_df).cumprod()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative quintile returns
colors_q = ['#d62728', '#ff7f0e', '#bcbd22', '#17becf', '#2ca02c']
for q, color in zip(range(1, 6), colors_q):
    ax1.plot(q_cum.index, q_cum[q], label=f'Q{q}', color=color, linewidth=1.8)
ax1.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
ax1.set_ylabel('Growth of $1')
ax1.set_title('Quintile Cumulative Returns (Long-Only, No Cost)', fontweight='bold')
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Average monthly returns by quintile
q_means = q_df.mean() * 100
ax2.bar([f'Q{q}' for q in range(1, 6)], q_means.values,
        color=colors_q, edgecolor='black', linewidth=0.5)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Avg Monthly Return (%)')
ax2.set_title('Average Monthly Return by Quintile', fontweight='bold')
for i, (bar, v) in enumerate(zip(ax2.patches, q_means.values)):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.01 if v>=0 else v-0.03,
             f'{v:.3f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../results/quintile_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAverage monthly returns by quintile:")
print(q_means.to_string())
print(f"\nQ5-Q1 monthly spread: {(q_means.iloc[4] - q_means.iloc[0]):.4f}%")


## 7. Rolling Performance — 12-Month Sharpe


In [ ]:
roll_win = 12
roll_sharpe_gross = (
    gross_returns.rolling(roll_win).mean() * 12 /
    (gross_returns.rolling(roll_win).std() * np.sqrt(12))
)
roll_sharpe_net = (
    net_returns.rolling(roll_win).mean() * 12 /
    (net_returns.rolling(roll_win).std() * np.sqrt(12))
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(roll_sharpe_gross.index, roll_sharpe_gross.values, label='Gross', color='#1f77b4', linewidth=1.8)
ax.plot(roll_sharpe_net.index,   roll_sharpe_net.values,   label='Net',   color='#2ca02c', linewidth=1.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(1, color='grey', linestyle='--', linewidth=0.8, label='Sharpe = 1.0')
ax.set_ylabel('Rolling 12-Month Annualized Sharpe')
ax.set_title('Rolling 12-Month Sharpe — Gross vs. Net Strategy', fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../results/rolling_sharpe_strategy.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Complete Metrics Summary Table


In [ ]:
print("\n" + "=" * 65)
print("FINAL PERFORMANCE SUMMARY — Long/Short Dollar-Neutral Strategy")
print("=" * 65)
print(metrics_df.round(3).to_string())
print("=" * 65)
print(f"Period: {gross_returns.index[0].date()} → {gross_returns.index[-1].date()}")
print(f"Universe: {'Synthetic (illustrative)' if SYNTHETIC else 'Live data'}")
print(f"Transaction cost assumption: {ONE_WAY_COST_BPS} bps one-way")
print(f"Rebalance: Monthly")
print()
print("SURVIVORSHIP BIAS WARNING:")
print("  Results are based on a static universe proxy and are NOT")
print("  representative of investable live performance. Point-in-time")
print("  constituent data (CRSP/Compustat via WRDS) is required for")
print("  production-grade backtests.")
print("=" * 65)


## 9. Honest Assessment

**What worked**:
- The quintile spread is likely positive and monotonic — confirming the composite
  signal has directional predictive power over the backtest period
- Transaction cost drag at 5 bps one-way is modest for a monthly-rebalance strategy

**Limitations that prevent taking these results at face value**:

| Issue | Impact |
|-------|--------|
| Survivorship bias (static universe) | Inflates returns by 1–3% annualized (estimated) |
| IC weights estimated in-sample | Overestimates composite signal quality |
| No short-selling constraints | L/S portfolio assumes equal ability to short |
| No market impact model | 5 bps is conservative for liquid names but inadequate for small-caps |
| Parameter choices not cross-validated | Monthly rebalance, quintile sort — tested once |
| Single data source (yfinance/synthetic) | No data quality validation vs. CRSP |

**Conclusion**: A positive net Sharpe (if observed) is encouraging but not sufficient
to conclude the strategy is investable. The next step is Notebook 06 — performance
attribution to determine what fraction of returns is genuine alpha vs. factor exposure.


In [ ]:
print("=" * 60)
print("Notebook 05 complete.")
print(f"Synthetic data: {SYNTHETIC}")
print("Outputs saved to ../results/")
print("=" * 60)
